# Trích embedding trên Colab GPU
1. Vào **Runtime → Change runtime type → T4 GPU** trước khi chạy (nếu không, ô kiểm tra GPU sẽ báo lỗi).
2. Notebook clone code từ GitHub, nên hãy `git push` các thay đổi mới nhất trước.
3. Chạy lần lượt từng ô.

In [ ]:
import shutil, subprocess
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Chưa bật GPU: Runtime → Change runtime type → T4 GPU, rồi chạy lại.')
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)

In [ ]:
!rm -rf /content/app && git clone https://github.com/Tuyen2k47/face_detect_doan.git /content/app
%cd /content/app
!git log --oneline -1

InsightFace kéo theo gói `onnxruntime` bản CPU, nên gỡ nó rồi cài lại `onnxruntime-gpu` để GPU được dùng.

In [ ]:
!pip install -q insightface==2.0 opencv-python-headless
!pip uninstall -y -q onnxruntime
!pip install -q --force-reinstall --no-deps onnxruntime-gpu==1.23.2
# cuDNN có sẵn trên Colab (9.27) làm conv lỗi trên T4 và ORT âm thầm chạy CPU.
!pip install -q nvidia-cudnn-cu12==9.10.2.21

In [ ]:
import onnxruntime as ort
providers = ort.get_available_providers()
print(providers)
assert 'CUDAExecutionProvider' in providers, 'onnxruntime-gpu chưa nhận CUDA; chạy lại ô cài đặt.'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Sửa đường dẫn dưới đây tới thư mục Drive chứa `<tên người>/*.mp4`.

In [ ]:
VIDEO_SRC = '/content/drive/MyDrive/video_data'
!rm -rf video_data && cp -r "$VIDEO_SRC" video_data
!ls video_data

In [ ]:
import os
# Bắt buộc dùng GPU trên Colab thay vì âm thầm fallback sang CPU.
os.environ['INSIGHTFACE_USE_CUDA'] = 'require'
!python preprocessing.py 2>&1 | tee preprocessing_log.txt

In [ ]:
from google.colab import files
for f in ['face_embeddings.npz', 'face_prototypes.npz', 'preprocessing_log.txt']:
    files.download(f)